# semantic_search/04a — Cancer-type baseline

Trains only the non-text reference space, `cancer_type_baseline`: one-hot cancer type and nothing else, through the same nested-CV XGBoost as `04_predict`. It answers how much of each target is explained by diagnosis alone — an embedding space that does not beat it has not shown it reads anything beyond cancer type.

**Runs after** `04_predict` has finished its `concat` setups. The baseline is only comparable to `concat` on the *same* patients and folds, so each run passes `--spaces concat cancer_type_baseline --cohort-mode common`: the `concat` features define the shared cohort, and the completed `concat` artifacts are reused rather than retrained. Targets whose `concat` setup is not yet complete in a window are skipped for that window, so this notebook never trains an embedding model.

The `cancer_type` target is excluded: the baseline features are that target's own labels.

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
import time
from pathlib import Path

from IPython.display import display


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "config.py").is_file() and (candidate / "semantic_search").is_dir():
            return candidate
    raise RuntimeError(f"Could not find repo root from {start}")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import config  # noqa: E402
from semantic_search import common  # noqa: E402


def check_inputs(preconditions: list[tuple[str, str]]) -> list[str]:
    """Report presence of each (label, path). Returns missing labels; never raises."""
    missing = []
    for label, path in preconditions:
        ok = os.path.exists(path)
        if not ok:
            missing.append(label)
        print(f"[{'ok ' if ok else 'MISSING'}] {label:<30} {path}")
    message = "All inputs present." if not missing else f"{len(missing)} missing: {', '.join(missing)}"
    print(f"\n{message}")
    return missing


def run_module(module: str, args: list[str] | None = None) -> int:
    """Run a module as a subprocess and stream its output."""
    cmd = [sys.executable, "-m", module] + (args or [])
    print("$ " + " ".join(cmd) + "\n", flush=True)
    t0 = time.time()
    proc = subprocess.run(cmd, cwd=REPO_ROOT)
    print(f"\nexit={proc.returncode}  elapsed={(time.time() - t0) / 60:,.1f} min", flush=True)
    return proc.returncode


print(f"repo root:  {REPO_ROOT}")
print(f"data root:  {config.DATA_PATH}")
print(f"this arm:   {config.SEMANTIC_SEARCH_PATH}")

## Configuration

`TARGETS` defaults to every registered target except `cancer_type`. CV settings must match the `04_predict` run whose `concat` artifacts are being paired with; otherwise the `concat` run signature changes and the script refuses to reuse them.

In [ ]:
MODULE = "semantic_search.train_prediction_models"

from semantic_search.prediction_targets import TARGETS as ALL_TARGETS  # noqa: E402

TARGETS = [t for t in ALL_TARGETS if t != "cancer_type"]
BASELINE = common.BASELINE_SPACE
PAIRED = common.FULL_SPACE_SOURCE  # "concat": defines the shared cohort
WINDOWS = common.DEFAULT_WINDOWS
MODELS = ["xgboost"]

# Must match 04_predict so the paired concat artifacts are reused, not refused.
COHORT_MODE = "common"
TREATMENT_GRANULARITY = "category"
MIN_TREATMENT_CLASS_N = 25
OUTER_FOLDS = 5
INNER_FOLDS = 3
SEED = 1234
N_JOBS = -1
AVPC_NEPC_LABELS = config.AVPC_NEPC_LABELS_PATH
OVERWRITE = False  # replaces baseline AND the paired concat artifacts; leave off
RUN_MODELS = True

print(f"targets:  {TARGETS}")
print(f"windows:  {WINDOWS}")
print(f"baseline: {BASELINE}  (paired with {PAIRED}, cohort mode {COHORT_MODE})")
print(f"CV:       {OUTER_FOLDS} outer x {INNER_FOLDS} inner; n_jobs={N_JOBS} ({os.cpu_count()} CPUs visible)")

## Preconditions

The baseline is built in memory from `cancer_type_df.csv.gz`. The `concat` feature files are still needed to define the shared cohort.

In [ ]:
missing = check_inputs(
    [("cancer-type labels", os.path.join(config.FEATURE_PATH, "cancer_type_df.csv.gz"))]
    + [(f"features {PAIRED}/{window}", common.feature_path(PAIRED, window)) for window in WINDOWS]
)

## Run plan

A target runs in a window only when its `concat` setup there is complete. Existing baseline setups on the same cohort are reused.

In [ ]:
def artifact_paths(target: str, space: str, window: str, model: str) -> list[str]:
    stem = f"{target}__{space}__{window}__{model}"
    return [
        os.path.join(common.PREDICTIONS_DIR, f"{stem}.parquet"),
        os.path.join(common.MODELS_DIR, f"{stem}.joblib"),
        os.path.join(common.PREDICTION_META_DIR, f"{stem}.json"),
    ]


def complete(target: str, space: str, window: str) -> bool:
    return all(
        os.path.exists(path)
        for model in MODELS
        for path in artifact_paths(target, space, window, model)
    )


plan = {}
for window in WINDOWS:
    plan[window] = [t for t in TARGETS if complete(t, PAIRED, window)]
    skipped = [t for t in TARGETS if t not in plan[window]]
    done = [t for t in plan[window] if complete(t, BASELINE, window)]
    print(f"[{window}] {len(plan[window])} / {len(TARGETS)} targets have {PAIRED} complete; "
          f"{len(done)} baseline setups already complete")
    if skipped:
        print(f"  skipped (run 04_predict first): {skipped}")

## Run baseline

One call per window, restricted to that window's runnable targets.

In [ ]:
common_args = [
    "--spaces", PAIRED, BASELINE,
    "--models", *MODELS,
    "--cohort-mode", COHORT_MODE,
    "--treatment-granularity", TREATMENT_GRANULARITY,
    "--min-treatment-class-n", str(MIN_TREATMENT_CLASS_N),
    "--outer-folds", str(OUTER_FOLDS),
    "--inner-folds", str(INNER_FOLDS),
    "--seed", str(SEED),
    "--n-jobs", str(N_JOBS),
    "--avpc-nepc-labels", str(AVPC_NEPC_LABELS),
]
if OVERWRITE:
    common_args.append("--overwrite")

if RUN_MODELS:
    for window, targets in plan.items():
        if not targets:
            print(f"[{window}] nothing to run")
            continue
        rc = run_module(MODULE, ["--targets", *targets, "--windows", window, *common_args])
        if rc != 0:
            print(
                f"\n[{window}] failed — see the traceback above. A 'different cohort or CV "
                "configuration' error means the settings above do not match the 04_predict "
                "run that built the concat artifacts."
            )
else:
    print("RUN_MODELS = False; skipped training and continuing to existing results.")

## Baseline vs concat

Pooled out-of-fold metrics on the shared cohort. `delta` is `concat − baseline`: how much the embeddings add beyond cancer type.

In [ ]:
import polars as pl

summary_path = common.result_path("prediction_metrics_summary")
if os.path.exists(summary_path):
    summary = pl.read_csv(summary_path).filter(
        pl.col("target").is_in(TARGETS)
        & pl.col("space").is_in([PAIRED, BASELINE])
        & pl.col("window").is_in(WINDOWS)
        & pl.col("model").is_in(MODELS)
    )
    keys = ["target", "window", "model"]
    for metric in ["macro_ovr_auc_pooled_oof", "macro_f1_pooled_oof"]:
        wide = (
            summary.select(*keys, "space", metric)
            .pivot(on="space", index=keys, values=metric)
        )
        if {PAIRED, BASELINE} <= set(wide.columns):
            wide = wide.with_columns(
                (pl.col(PAIRED) - pl.col(BASELINE)).alias("delta")
            ).sort(keys)
        print(metric)
        display(wide)
    display(
        summary.filter(pl.col("space") == BASELINE)
        .select(*keys, "n_patients", "n_transformed_features", "n_classes")
        .sort(keys)
    )
else:
    print(f"No prediction summary at {summary_path}")